# Template Quest Notebook

In [1]:
# Import neccessary modules, add to this cell as needed
# Provided PySpark examples

import os
import subprocess

# 1. Force Python to find and lock Java 17 FIRST
try:
    java_17_path = subprocess.check_output(['/usr/libexec/java_home', '-v', '17']).decode('utf-8').strip()
    os.environ['JAVA_HOME'] = java_17_path
    print(f"Successfully locked JAVA_HOME to: {os.environ['JAVA_HOME']}")
except Exception as e:
    print(f"Failed! The exact error is: {e}")

# 2. Fix the Mac hostname binding issue for Spark
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# 3. THEN import PySpark so it is forced to use the new environment
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

Successfully locked JAVA_HOME to: /Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home


## Part 1: Load the Sample Dataset

In [2]:
# Initiate a new Spark session and set the case sensitivity option
spark = (
    SparkSession.builder
        .appName("cyberquest")
        .getOrCreate()
)
spark.conf.set("spark.sql.caseSensitive", True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/controlroom/Documents/RearcProject/cyber-quest/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/19 12:46:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# TODO: Load the raw data
df_bronze = spark.read.json("data/")

# Show the first 5 rows without cutting off the text
df_bronze.show(5, truncate=False)

+----------------------------------------------+-----------+---------+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
# TODO: Parse the raw data into something relevant and usable
# We extract specific fields from the JSON string inside the '_raw' column using F.get_json_object
df_silver = df_bronze.select(
    F.col("Computer"),
    F.col("EventCode"),
    F.get_json_object(F.col("_raw"), "$.UtcTime").alias("UtcTime"),
    F.get_json_object(F.col("_raw"), "$.ProcessId").alias("ProcessId"),
    F.get_json_object(F.col("_raw"), "$.Image").alias("Image"),
    F.get_json_object(F.col("_raw"), "$.QueryName").alias("QueryName")
)

df_silver.createOrReplaceTempView("sysmon_silver")

In [5]:
# Provided PySpark Example
df_silver.limit(5).show()

+--------------------+---------+--------------------+---------+--------------------+---------+
|            Computer|EventCode|             UtcTime|ProcessId|               Image|QueryName|
+--------------------+---------+--------------------+---------+--------------------+---------+
|win-host-ctus-att...|       23|2023-01-27 11:19:...|     3764|C:\Program Files\...|     NULL|
|win-dc-ctus-attac...|       23|2023-01-27 11:19:...|     3928|C:\Program Files\...|     NULL|
|win-host-ctus-att...|       23|2023-01-27 11:19:...|     1880|C:\Program Files\...|     NULL|
|win-dc-ctus-attac...|       10|2023-01-27 11:19:...|     NULL|                NULL|     NULL|
|win-dc-ctus-attac...|       10|2023-01-27 11:19:...|     NULL|                NULL|     NULL|
+--------------------+---------+--------------------+---------+--------------------+---------+



In [6]:
# Provided PySpark SQL Example
spark.sql("""
SELECT *
FROM sysmon_silver
LIMIT 5
""").show()

+--------------------+---------+--------------------+---------+--------------------+---------+
|            Computer|EventCode|             UtcTime|ProcessId|               Image|QueryName|
+--------------------+---------+--------------------+---------+--------------------+---------+
|win-host-ctus-att...|       23|2023-01-27 11:19:...|     3764|C:\Program Files\...|     NULL|
|win-dc-ctus-attac...|       23|2023-01-27 11:19:...|     3928|C:\Program Files\...|     NULL|
|win-host-ctus-att...|       23|2023-01-27 11:19:...|     1880|C:\Program Files\...|     NULL|
|win-dc-ctus-attac...|       10|2023-01-27 11:19:...|     NULL|                NULL|     NULL|
|win-dc-ctus-attac...|       10|2023-01-27 11:19:...|     NULL|                NULL|     NULL|
+--------------------+---------+--------------------+---------+--------------------+---------+



## Part 2: Detection Engineering

In [7]:
# Part 2: Detection Engineering
# Hypothesis: Microsoft Office applications making potentially malicious web calls (DNS Queries)

detection_query = """
SELECT 
    UtcTime,
    Computer,
    ProcessId,
    Image AS Initiating_Process,
    QueryName AS Domain_Queried
FROM sysmon_silver
WHERE EventCode = '22'
  AND (
      lower(Image) LIKE '%winword.exe' OR 
      lower(Image) LIKE '%excel.exe' OR 
      lower(Image) LIKE '%powerpnt.exe'
  )
"""

# Execute the query and store it in a new dataframe
df_detection = spark.sql(detection_query)

# Show the results of our detection!
print("Detection Results: Suspicious Office DNS Queries")
df_detection.show(truncate=False)

Detection Results: Suspicious Office DNS Queries
+-----------------------+------------------------------+---------+-----------------------------------------------------------+---------------------------------+
|UtcTime                |Computer                      |ProcessId|Initiating_Process                                         |Domain_Queried                   |
+-----------------------+------------------------------+---------+-----------------------------------------------------------+---------------------------------+
|2023-01-27 11:30:14.523|win-host-ctus-attack-range-212|4200     |C:\Program Files\Microsoft Office\root\Office16\WINWORD.EXE|www.mediafire.com                |
|2023-01-27 11:30:13.961|win-host-ctus-attack-range-212|4200     |C:\Program Files\Microsoft Office\root\Office16\WINWORD.EXE|support.content.office.net       |
|2023-01-27 11:30:13.957|win-host-ctus-attack-range-212|4200     |C:\Program Files\Microsoft Office\root\Office16\WINWORD.EXE|ecs.office.com      

## Part 3: Additional Steps

In [8]:
import uuid
from pyspark.sql.types import StringType
import pyspark.sql.functions as F

### Part 3.1: Normalization

In [9]:
# Mapping the raw Sysmon fields to the standard Elastic Common Schema (ECS)
df_normalized = df_detection.select(
    F.col("UtcTime").alias("@timestamp"),
    F.col("Computer").alias("host.name"),
    F.col("Initiating_Process").alias("process.executable"),
    F.col("ProcessId").alias("process.pid"),
    F.col("Domain_Queried").alias("dns.question.name")
)

### Part 3.2: Alert Table

In [10]:
# Creating a mock Threat Intel lookup function
def check_threat_intel(domain):
    if domain is None:
        return "UNKNOWN"
    domain = domain.lower()
    
    # Flag the specific suspicious domain we found in Part 2
    if "mediafire.com" in domain:
        return "MALICIOUS - Known file sharing site accessed by Office"
    elif "office.com" in domain or "office.net" in domain:
        return "BENIGN - Microsoft Infrastructure"
    else:
        return "UNKNOWN"

# Register the Python function as a PySpark UDF (User Defined Function)
ti_udf = F.udf(check_threat_intel, StringType())
uuid_udf = F.udf(lambda: str(uuid.uuid4()), StringType())

/Users/controlroom/Documents/RearcProject/cyber-quest/.venv/lib/python3.11/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/Users/controlroom/Documents/RearcProject/cyber-quest/.venv/lib/python3.11/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


### Part 3.3: Enrichment

In [12]:
# Adding metadata an analyst would need for triage on a dashboard
df_alert = df_normalized \
    .withColumn("alert_id", uuid_udf()) \
    .withColumn("rule_name", F.lit("Suspicious DNS Query by MS Office Application")) \
    .withColumn("mitre_technique", F.lit("T1566.001 - Phishing: Spearphishing Attachment")) \
    .withColumn("threat_intel", ti_udf(F.col("`dns.question.name`"))) \
    .withColumn("severity", F.when(F.col("threat_intel").like("MALICIOUS%"), "HIGH").otherwise("LOW"))

print("Final Packaged Alerts for SOC Analysts:")
# Filtering to only show the HIGH severity alerts (the true positives) to the analyst
df_alert.filter(F.col("severity") == "HIGH").show(truncate=False)

Final Packaged Alerts for SOC Analysts:


+-----------------------+------------------------------+-----------------------------------------------------------+-----------+-----------------+------------------------------------+---------------------------------------------+----------------------------------------------+------------------------------------------------------+--------+
|@timestamp             |host.name                     |process.executable                                         |process.pid|dns.question.name|alert_id                            |rule_name                                    |mitre_technique                               |threat_intel                                          |severity|
+-----------------------+------------------------------+-----------------------------------------------------------+-----------+-----------------+------------------------------------+---------------------------------------------+----------------------------------------------+------------------------------------------

## Summary

**Methodology:**
1. **Data Parsing:** Extracted key fields (`Computer`, `EventCode`, `Image`, `QueryName`, `UtcTime`, `ProcessId`) from raw Sysmon JSON logs using PySpark's `get_json_object` to create a clean, usable dataframe.
2. **Detection Engineering:** Built a detection query targeting Event Code 22 (DNS queries). I filtered the data for Microsoft Office applications (`winword.exe`, `excel.exe`, `powerpnt.exe`) to test the phishing hypothesis. This isolated an anomalous query from `WINWORD.EXE` to `www.mediafire.com`, strongly indicating a malicious document attempting to download a secondary payload.
3. **Normalization & Alerting:** Normalized the resulting dataframe to the Elastic Common Schema (ECS) (e.g., mapping `Computer` to `host.name`). I then utilized a PySpark UDF to simulate Threat Intelligence enrichment, flagging the file-sharing domain as malicious, and packaged the data into an actionable alert with MITRE ATT&CK mapping (T1566.001).

**AI Usage Disclosure:**
I utilized AI as a reference and pair-programming tool during this exercise. It primarily helped with setting up the local Mac environment (troubleshooting Java 17 pathing and PySpark localhost binding). Additionally, it served as a syntax reference for PySpark UDF creation and for troubleshooting an unresolved column error caused by dots in the ECS schema names.